# 均线版：仅 Stage 11
MA_DIRECTION_REFERENCE / NO V2 / ZERO。原 Stage 8 不中断，原任务继续 Stage 9。本文件必须使用独立 CPU 运行时。

## 1. 安装依赖

In [ ]:
%pip -q install numpy==2.3.5 pandas==3.0.1 psutil==7.0.0

## 2. 挂载 Drive、上传一次代码包并恢复本阶段
没有 GitHub 令牌。行情原始 ZIP 不在代码包内，缺少的行情按原校验规则下载。

In [ ]:
STAGE=11
from google.colab import drive, files
from pathlib import Path
import os, sys, shutil, hashlib, zipfile, json, subprocess, time, queue, threading, tempfile
drive.mount('/content/drive')
MAIN=Path('/content/drive/MyDrive/btc_eth_ma_direction_optimizations/ec73e9c/MA_AUDITED_RESUME_cad3c8c')
PACKAGE=Path('/content/drive/MyDrive/btc_eth_ma_parallel/ma_parallel_code_cad3c8c.zip')
EXPECTED='367a13943e47d07dd64c76dd662db5abc0ba414b1edeb8985b29c09a4d9c273d'
if not PACKAGE.is_file():
    print('请选择下载的 ma_parallel_code_cad3c8c.zip；只需上传一次。', flush=True)
    uploaded=files.upload()
    assert len(uploaded)==1
    name,content=next(iter(uploaded.items()))
    assert hashlib.sha256(content).hexdigest()==EXPECTED, '代码包校验失败'
    PACKAGE.parent.mkdir(parents=True,exist_ok=True)
    PACKAGE.write_bytes(content)
assert hashlib.sha256(PACKAGE.read_bytes()).hexdigest()==EXPECTED, 'Drive代码包校验失败'
REPO=Path('/content/ma_parallel_code_cad3c8c')
with zipfile.ZipFile(PACKAGE) as z:
    manifest=json.loads(z.read('PARALLEL_BUNDLE_MANIFEST.json'))
    for name,expected in manifest['files'].items():
        assert hashlib.sha256(z.read(name)).hexdigest()==expected, name
    for name in z.namelist():
        assert not Path(name).is_absolute() and '..' not in Path(name).parts
    z.extractall(REPO)
RESULTS=Path(f'/content/ma_parallel_stage{STAGE}_results')
DRIVE_RESULTS=MAIN/f'parallel_stage{STAGE}'
DATA=Path('/content/ma_parallel_market_cache')
RESULTS.mkdir(parents=True,exist_ok=True); DATA.mkdir(parents=True,exist_ok=True)
if DRIVE_RESULTS.exists():
    shutil.copytree(DRIVE_RESULTS,RESULTS,dirs_exist_ok=True,ignore=shutil.ignore_patterns('*.tmp','*.part','*.copying'))
# 每个运行时只写自己的本地行情缓存；不并发改写共享行情。
for folder in ['native','native_extended_20261115']:
    source=Path('/content/drive/MyDrive/btc_eth_market_cache')/folder
    if source.is_dir(): shutil.copytree(source,DATA/folder,dirs_exist_ok=True)
os.environ.update(BTC_REPO_ROOT=str(REPO),BTC_OUTPUT_ROOT=str(RESULTS),BTC_DATA_ROOT=str(DATA),BTC_DRIVE_RESULTS=str(DRIVE_RESULTS))
os.environ.pop('BTC_DRIVE_DATA',None)
def visible(cmd,env=None):
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    messages=queue.Queue()
    def read():
        for line in p.stdout: messages.put(line)
        messages.put(None)
    threading.Thread(target=read,daemon=True).start()
    while True:
        try: line=messages.get(timeout=30)
        except queue.Empty:
            print('仍在运行：准备数据或计算中',flush=True);continue
        if line is None:break
        print(line,end='',flush=True)
    assert p.wait()==0, '任务停止；保留当前运行时与输出，查看上方具体错误。'
CMD=[sys.executable,'-u',str(REPO/'outputs/ma_parallel/runner.py'),'--stage',str(STAGE),'--main-drive',str(MAIN)]
print('代码包校验通过，阶段',STAGE,flush=True)


## 3. 准备接回原任务（不启动回测）
两个新 Notebook 都运行到这里，再到原 Colab 的左侧「终端」运行下一段命令。

In [ ]:
visible(CMD+['--setup-only'])
control=MAIN/'parallel_control';control.mkdir(exist_ok=True)
for name in ['install_handoff.py','delegation_guard.py']:
    content=(REPO/'outputs/ma_parallel'/name).read_bytes()
    dest=control/name
    if dest.exists(): assert dest.read_bytes()==content, '已有其他版本接回脚本'
    else:
        with tempfile.NamedTemporaryFile(dir=control,delete=False) as f:
            f.write(content);temp=Path(f.name)
        temp.replace(dest)
print('SETUP_READY。两个 Notebook 均准备好后，在原运行时终端执行：')
print('python '+str(control/'install_handoff.py'))


## 4. 原运行时终端执行一次
不要停止原 Stage 8，不要向正在忙的原内核排队代码格。打开原 Colab 左侧「终端」，粘贴：
```bash
python /content/drive/MyDrive/btc_eth_ma_direction_optimizations/ec73e9c/MA_AUDITED_RESUME_cad3c8c/parallel_control/install_handoff.py
```
必须看到 `HANDOFF_INSTALLED` 后再运行下面第 5 格。若提示找不到原协调进程，先保留现场，不能声称已接回。

## 5. 只运行 Stage 11
中断后恢复本 Notebook 的 1、2、3、5；已完成账户跳过，未完成账户按原断点机制恢复。

In [ ]:
assert (MAIN/'delegate_stage10.json').exists() and (MAIN/'delegate_stage11.json').exists()
visible(CMD)

## 6. 完成位置
原任务会在完成 Stage 9 后等待并校验导入这两组结果，不重复回放 Stage 10/11。三个运行时都完成后，在原 Notebook 运行第 6 节只读汇总。

In [ ]:
print('本阶段结果：',DRIVE_RESULTS)
state=DRIVE_RESULTS/'parallel_state.json'
print(state.read_text() if state.exists() else '尚未生成状态')